# Eksperimen ABSA IndoBERT — dataset_fuiyohh

Notebook ini menjalankan validasi, preprocessing, multilabel-stratified split, training, evaluasi, dan pencatatan MLflow. Pilih runtime GPU (T4 atau yang lebih baik) sebelum menjalankan semua cell. MLflow lokal dipakai secara default, sehingga tunnel dan kredensial S3 tidak wajib.

In [ ]:
MOUNT_DRIVE = True

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
import os

REPO_URL = 'https://github.com/cupskii/Model_ABSA.git'
REPO_DIR = '/content/Model_ABSA'

if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only

%cd {REPO_DIR}

In [ ]:
%pip install -q -r requirements.txt

import os

_RESTART_MARKER = '/content/.absa_dependencies_ready'
if not os.path.exists(_RESTART_MARKER):
    with open(_RESTART_MARKER, 'w', encoding='utf-8') as marker:
        marker.write('1')
    print('Dependency terpasang. Runtime akan restart satu kali.')
    print('Sesudah restart, pilih Runtime > Run all lagi.')
    os.kill(os.getpid(), 9)
else:
    print('Dependency siap.')

## Konfigurasi data dan MLflow

Jika dataset belum ada dalam repository hasil clone, notebook akan membuka dialog upload. Untuk MLflow remote, buat Colab Secret bernama `MLFLOW_TRACKING_URI`; jika tidak ada, metadata tracking disimpan di SQLite lokal `mlflow_fuiyohh.db`. Bila server remote memakai object storage langsung, tambahkan juga Secrets AWS/S3 yang digunakan server tersebut.

In [ ]:
import os
from pathlib import Path

os.chdir('/content/Model_ABSA')
CONFIG_PATH = 'configs/experiment_indobert_fuiyohh.yaml'
DATASET_PATH = Path('data/raw/dataset_fuiyohh.csv')

if not DATASET_PATH.is_file():
    from google.colab import files
    print('Pilih file dataset_fuiyohh.csv dari komputer Anda.')
    uploaded = files.upload()
    if 'dataset_fuiyohh.csv' not in uploaded:
        raise FileNotFoundError('File yang dipilih harus bernama dataset_fuiyohh.csv')
    DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)
    DATASET_PATH.write_bytes(uploaded['dataset_fuiyohh.csv'])

try:
    from google.colab import userdata
    remote_tracking_uri = userdata.get('MLFLOW_TRACKING_URI')
except Exception:
    remote_tracking_uri = None

if remote_tracking_uri:
    for secret_name in ('AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY', 'MLFLOW_S3_ENDPOINT_URL', 'AWS_DEFAULT_REGION'):
        try:
            secret_value = userdata.get(secret_name)
        except Exception:
            secret_value = None
        if secret_value:
            os.environ[secret_name] = secret_value
    os.environ['MLFLOW_TRACKING_URI'] = remote_tracking_uri
    print('MLflow remote:', remote_tracking_uri)
else:
    os.environ.pop('MLFLOW_TRACKING_URI', None)
    print('MLflow lokal: sqlite:///mlflow_fuiyohh.db')

print('Config :', CONFIG_PATH)
print('Dataset:', DATASET_PATH, f'({DATASET_PATH.stat().st_size:,} bytes)')

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU tidak tersedia. Pilih Runtime > Change runtime type > T4 GPU.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from run_experiment import run_experiment

metrics = run_experiment(CONFIG_PATH)
metrics

Checkpoint terbaik otomatis disalin oleh pipeline ke `MyDrive/absa_models/absa_indobert/dataset_fuiyohh_baseline/` ketika Google Drive di-mount. Cell berikut juga menyalin database dan artefak MLflow lokal agar riwayat eksperimen tidak hilang saat runtime berakhir.

In [ ]:
import os
import shutil

local_mlruns = '/content/Model_ABSA/mlruns'
local_mlflow_db = '/content/Model_ABSA/mlflow_fuiyohh.db'
if MOUNT_DRIVE and os.path.isdir('/content/drive/MyDrive') and os.path.isfile(local_mlflow_db):
    destination = '/content/drive/MyDrive/absa_models/mlflow_fuiyohh'
    os.makedirs(destination, exist_ok=True)
    shutil.copy2(local_mlflow_db, destination)
    if os.path.isdir(local_mlruns):
        shutil.copytree(local_mlruns, os.path.join(destination, 'mlruns'), dirs_exist_ok=True)
    print('Riwayat MLflow disalin ke:', destination)
else:
    print('Tidak ada MLflow lokal yang perlu disalin.')